# AI Home Mixologist - Qwen Grounded Generation RAG

## Google Colab Teaching Notebook

This notebook builds an English-language cocktail assistant step by step.

The project demonstrates:

- loading and cleaning a cocktail dataset
- matching available home ingredients to recipes
- converting recipes into complete recipe documents
- creating embeddings
- building a FAISS vector index
- manually implementing retrieval-augmented generation, or RAG
- using FAISS-retrieved evidence for descriptive cocktail preferences
- prompting Qwen Instruct to produce a visible grounded explanation from retrieved evidence
- keeping deterministic recipe cards below the generated explanation for fact checking
- evaluating retrieval and recommendation behaviour

**Notebook rule:** all project code stays inside this one notebook. No separate Python files are required.


# Section 1: Install Packages

Before we can build the mixologist system, we need to install and verify the Python libraries used throughout the notebook.

In later sections, we will use these packages for data handling, ingredient matching, embeddings, vector search, and LLM recommendations.

This section is intentionally simple because package setup should be easy to rerun in Google Colab.


## Why This Step Is Necessary

- `pandas` and `numpy` help us load, clean, and analyze recipe data.
- `datasets` lets us download the Hugging Face cocktail dataset directly in Colab.
- `sentence-transformers` turns recipe documents into embeddings.
- `faiss-cpu` lets us search embeddings efficiently.
- `transformers` and `accelerate` let us run a small teaching LLM in Colab.
- `ipywidgets` helps us create a simple interactive demo.

We will implement the RAG pipeline manually instead of using LangChain so each concept is visible and understandable.

In [ ]:
# ============================================================
# Section 1: Install Packages
# ============================================================

# Install the libraries needed for the full notebook.
# The -q flag keeps the Colab output cleaner for beginners.
%pip install -q pandas numpy datasets sentence-transformers faiss-cpu transformers accelerate bitsandbytes gradio ipywidgets

# Import package metadata so we can confirm versions after installation.
import importlib.metadata as package_metadata

# Map display names to installed package names.
required_packages = {
    "pandas": "pandas",
    "numpy": "numpy",
    "datasets": "datasets",
    "sentence-transformers": "sentence-transformers",
    "faiss-cpu": "faiss-cpu",
    "transformers": "transformers",
    "accelerate": "accelerate",
    "bitsandbytes": "bitsandbytes",
    "gradio": "gradio",
    "ipywidgets": "ipywidgets",
}

print("Installed package versions:")

for display_name, package_name in required_packages.items():
    try:
        package_version = package_metadata.version(package_name)
    except package_metadata.PackageNotFoundError:
        package_version = "installed but version metadata is unavailable"
    print(f"- {display_name}: {package_version}")

print("\nAll required packages are ready.")


## Short Comments

- We install packages first so every later section can run without missing-library errors.
- Version checks make the notebook easier to debug if something behaves differently in the future.
- Keeping installation in one cell makes the notebook clean and beginner-friendly.
- No LangChain is installed because this notebook builds the RAG pipeline manually.

# Section 2: Load Dataset

In this section, we load the cocktail recipe dataset from Hugging Face.

We will use the dataset ID provided for this project:

```text
erwanlc/cocktails_recipe
```

The goal is to create one raw DataFrame called `cocktails_raw_df`. Later sections will explore, clean, and transform this DataFrame into a recommendation-ready format.


## Why This Step Is Necessary

A recommendation system starts with data.

Before we can match ingredients, build documents, generate embeddings, or retrieve recipes, we need to load the recipe dataset into memory.

This section also prints the dataset features so beginners can see what columns are available before using them.


In [ ]:
# ============================================================
# Section 2: Load Dataset
# ============================================================

# Import libraries used for dataset loading and table handling.
from datasets import load_dataset
import pandas as pd


# Store the Hugging Face dataset ID and split in clear variables.
DATASET_ID = "erwanlc/cocktails_recipe"
DATASET_SPLIT = "train"


def load_huggingface_dataset(dataset_id, dataset_split):
    """Load a Hugging Face dataset split into memory."""
    return load_dataset(dataset_id, split=dataset_split)


def convert_dataset_to_dataframe(huggingface_dataset):
    """Convert a Hugging Face Dataset object into a pandas DataFrame."""
    return huggingface_dataset.to_pandas()


def print_loading_report(dataset_id, dataset_split, huggingface_dataset, dataframe):
    """Print a beginner-friendly summary of the loaded dataset."""
    print(f"Dataset ID: {dataset_id}")
    print(f"Dataset split: {dataset_split}")
    print(f"Rows: {dataframe.shape[0]}")
    print(f"Columns: {dataframe.shape[1]}")

    print("\nColumn names:")
    print(list(dataframe.columns))

    print("\nDataset features:")
    print(huggingface_dataset.features)


# Load the Hugging Face dataset split into the Colab environment.
cocktails_dataset = load_huggingface_dataset(DATASET_ID, DATASET_SPLIT)

# Convert the dataset to a pandas DataFrame for easier data analysis.
cocktails_raw_df = convert_dataset_to_dataframe(cocktails_dataset)

# Print a short loading report so we know the dataset is ready.
print_loading_report(DATASET_ID, DATASET_SPLIT, cocktails_dataset, cocktails_raw_df)

# Display the first few rows so we can visually confirm the data loaded correctly.
cocktails_raw_df.head()


## Short Comments

- We use Hugging Face `datasets` so the notebook can download the dataset directly in Colab.
- We print the features before cleaning so the data structure is transparent.
- We load the raw data into `cocktails_raw_df` and keep it unchanged for now.
- Cleaning happens later so we can compare the original data with the cleaned version.


# Section 3: Explore Dataset

Now that the dataset is loaded, we need to understand what it contains before cleaning or modeling.

Exploratory data analysis, often called EDA, helps us answer basic questions:

- How many recipes are in the dataset?
- What columns are available?
- Which columns have missing values?
- Are there duplicate cocktail names?
- What do the ingredient strings look like?
- Which glass types appear most often?

This section does not modify the dataset. It only studies `cocktails_raw_df`.


## Why This Step Is Necessary

A mixology recommendation system depends heavily on clean recipe and ingredient data.

If we skip exploration, we might make wrong assumptions about column names, missing values, duplicate recipes, or ingredient formatting.

By inspecting the raw dataset first, we can make better decisions in the cleaning section.


In [ ]:
# ============================================================
# Section 3: Explore Dataset
# ============================================================

# Import display so tables look clean in Google Colab.
from IPython.display import display

import pandas as pd


def show_dataset_shape(dataframe):
    """Print the number of rows and columns in a DataFrame."""
    row_count, column_count = dataframe.shape
    print(f"Rows: {row_count}")
    print(f"Columns: {column_count}")


def create_column_summary(dataframe):
    """Create a summary table for column types, missing values, and unique values."""
    summary_df = pd.DataFrame({
        "column_name": dataframe.columns,
        "data_type": [str(dataframe[column].dtype) for column in dataframe.columns],
        "missing_values": [dataframe[column].isna().sum() for column in dataframe.columns],
        "missing_percent": [dataframe[column].isna().mean() * 100 for column in dataframe.columns],
        "unique_values": [dataframe[column].nunique(dropna=True) for column in dataframe.columns],
    })

    summary_df["missing_percent"] = summary_df["missing_percent"].round(2)
    return summary_df


def count_duplicate_titles(dataframe, title_column="title"):
    """Count duplicate cocktail titles if the title column exists."""
    if title_column not in dataframe.columns:
        return None

    return dataframe[title_column].duplicated().sum()


def show_top_values(dataframe, column_name, top_n=10):
    """Show the most common values in one column."""
    if column_name not in dataframe.columns:
        print(f"Column not found: {column_name}")
        return

    top_values = dataframe[column_name].value_counts(dropna=False).head(top_n)
    display(top_values.to_frame(name="count"))


def calculate_text_lengths(dataframe, text_columns):
    """Calculate simple text length statistics for selected columns."""
    length_rows = []

    for column_name in text_columns:
        if column_name not in dataframe.columns:
            continue

        text_lengths = dataframe[column_name].fillna("").astype(str).str.len()

        length_rows.append({
            "column_name": column_name,
            "min_length": int(text_lengths.min()),
            "average_length": round(float(text_lengths.mean()), 2),
            "max_length": int(text_lengths.max()),
        })

    return pd.DataFrame(length_rows)


def show_sample_recipes(dataframe, sample_size=3):
    """Display a few recipes so we can inspect the raw format."""
    columns_to_show = [
        column_name
        for column_name in ["title", "glass", "garnish", "recipe", "ingredients"]
        if column_name in dataframe.columns
    ]

    display(dataframe[columns_to_show].head(sample_size))


# Confirm the raw dataset exists from Section 2.
if "cocktails_raw_df" not in globals():
    raise NameError("Please run Section 2 first so cocktails_raw_df is available.")


# 1. Show the basic dataset size.
print("Dataset shape")
show_dataset_shape(cocktails_raw_df)


# 2. Summarize each column.
print("\nColumn summary")
column_summary_df = create_column_summary(cocktails_raw_df)
display(column_summary_df)


# 3. Check for duplicate cocktail titles.
duplicate_title_count = count_duplicate_titles(cocktails_raw_df)
print("\nDuplicate title count:", duplicate_title_count)


# 4. Display a few raw recipe examples.
print("\nSample raw recipes")
show_sample_recipes(cocktails_raw_df, sample_size=3)


# 5. Show the most common glass types.
print("\nTop glass types")
show_top_values(cocktails_raw_df, column_name="glass", top_n=10)


# 6. Measure text lengths for fields that will later become recipe documents.
print("\nText length summary")
text_length_summary_df = calculate_text_lengths(
    cocktails_raw_df,
    text_columns=["title", "garnish", "recipe", "ingredients"],
)
display(text_length_summary_df)


## Short Comments

- We check shape first to understand the size of the project data.
- We inspect missing values because empty recipe fields can break matching and retrieval later.
- We check duplicate titles because duplicate recipes can bias recommendations.
- We inspect raw ingredient strings because Section 4 will need to parse them into clean ingredient lists.
- We measure text length because recipe, garnish, and ingredient fields will later become searchable documents.


# Section 4: Clean Dataset

In this section, we turn the raw cocktail data into a cleaner structure that later sections can use.

The most important cleaning task is parsing the `ingredients` column. In the Hugging Face dataset, each row stores ingredients as text that looks like a Python list of pairs:

```text
[['4.5 cl', 'Rutte Dry Gin'], ['2.25 cl', 'Lime juice (freshly squeezed)']]
```

We need to convert that text into real Python lists, separate the amount from the ingredient name, and create careful ingredient match keys.

This matters because not every `juice` is interchangeable. For example:

- `Passion fruit juice` should not match `lime juice` just because both contain the word `juice`.
- `Lemon juice (freshly squeezed)` should be treated as needing a fresh lemon, not bottled lemon juice.
- `Apple juice` should remain a fruit juice product.

## Why This Step Is Necessary

The raw dataset is useful for humans, but recommender systems need structured data.

Cleaning gives us:

- one recipe per row
- reliable text fields
- parsed ingredients
- display names for missing ingredients
- careful match keys for ingredient comparison
- a stable `recipe_id`

Later sections will use this cleaned data for ingredient matching, vector search, and recommendation prompts.

In [ ]:
# ============================================================
# Section 4: Clean Dataset
# ============================================================

# Import libraries for parsing text safely and normalizing words.
import ast
import re
import unicodedata

import pandas as pd


FRESH_PREPARATION_MARKERS = [
    "freshly squeezed",
    "fresh squeezed",
    "fresh pressed",
    "fresh press",
    "freshly pressed",
]

FRESH_FRUIT_WORDS = {
    "lemon", "lime", "orange", "grapefruit", "pineapple", "passion fruit",
    "apple", "cranberry", "tomato", "pomegranate", "watermelon",
}

FRUIT_DETAIL_WORDS = {
    "fresh", "freshly", "squeezed", "pressed", "sweetened", "unsweetened",
    "pink", "white", "red", "green", "chilled",
}

SPIRIT_AND_COMMON_ALIASES = {
    "gin": ["gin", "dry gin", "old tom gin", "sloe gin", "genever"],
    "rum": ["rum", "light rum", "white rum", "dark rum", "aged rum", "navy rum", "jamaican rum"],
    "vodka": ["vodka"],
    "tequila": ["tequila"],
    "mezcal": ["mezcal"],
    "whiskey": ["whiskey", "whisky", "bourbon", "scotch", "rye whiskey", "rye whisky"],
    "brandy": ["brandy", "cognac", "calvados", "armagnac"],
    "absinthe": ["absinthe"],
    "pisco": ["pisco"],
    "cachaca": ["cachaca"],
    "sugar syrup": ["sugar syrup", "simple syrup", "rich syrup"],
    "honey syrup": ["honey syrup"],
    "grenadine": ["grenadine", "grenadine syrup"],
    "mint leaves": ["mint", "mint leaves"],
    "soda water": ["soda water", "club soda", "carbonated water"],
    "angostura bitters": ["angostura bitters", "angostura aromatic bitters"],
    "orange bitters": ["orange bitters"],
    "triple sec": ["triple sec"],
    "orange curacao liqueur": ["orange curacao", "orange curacao liqueur", "blue curacao"],
}


def clean_text(value):
    """Convert a value into clean plain text."""
    if value is None:
        return ""

    if not isinstance(value, (list, tuple, dict)) and pd.isna(value):
        return ""

    cleaned_value = str(value).strip()
    cleaned_value = re.sub(r"\s+", " ", cleaned_value)
    return cleaned_value


def remove_accents(text):
    """Convert accented characters into simpler ASCII-like text."""
    normalized_text = unicodedata.normalize("NFKD", text)
    ascii_text = normalized_text.encode("ascii", "ignore").decode("ascii")
    return ascii_text


def normalize_ingredient_name(ingredient_name):
    """Normalize an ingredient name for matching and grouping."""
    ingredient_text = clean_text(ingredient_name).lower()
    ingredient_text = remove_accents(ingredient_text)

    # Remove parenthetical details such as "(freshly squeezed)".
    ingredient_text = re.sub(r"\([^)]*\)", " ", ingredient_text)

    # Replace punctuation with spaces, then compress extra spaces.
    ingredient_text = re.sub(r"[^a-z0-9]+", " ", ingredient_text)
    ingredient_text = re.sub(r"\s+", " ", ingredient_text).strip()
    return ingredient_text


def normalize_with_details(ingredient_name):
    """Normalize text while keeping useful parenthetical words such as freshly squeezed."""
    ingredient_text = clean_text(ingredient_name).lower()
    ingredient_text = remove_accents(ingredient_text)
    ingredient_text = re.sub(r"[^a-z0-9]+", " ", ingredient_text)
    ingredient_text = re.sub(r"\s+", " ", ingredient_text).strip()
    return ingredient_text


def contains_whole_phrase(text, phrase):
    """Check whether a phrase appears as a whole phrase inside normalized text."""
    return bool(re.search(rf"\b{re.escape(phrase)}\b", text))


def has_fresh_preparation_marker(raw_ingredient_name):
    """Detect whether a juice ingredient explicitly asks for fresh preparation."""
    normalized_with_details = normalize_with_details(raw_ingredient_name)
    return any(marker in normalized_with_details for marker in FRESH_PREPARATION_MARKERS)


def extract_fruit_before_juice(normalized_without_details):
    """Extract the fruit phrase before the word juice."""
    juice_match = re.search(r"\bjuice\b", normalized_without_details)

    if not juice_match:
        return None

    fruit_part = normalized_without_details[:juice_match.start()].strip()
    fruit_tokens = [
        token
        for token in fruit_part.split()
        if token not in FRUIT_DETAIL_WORDS
    ]
    fruit_name = " ".join(fruit_tokens).strip()
    return fruit_name or None


def canonicalize_alias_ingredient(normalized_without_details):
    """Map common broad ingredient names to stable keys without overmatching juices."""
    for canonical_name, aliases in SPIRIT_AND_COMMON_ALIASES.items():
        for alias in aliases:
            if contains_whole_phrase(normalized_without_details, alias):
                return canonical_name

    return None


def create_ingredient_match_key(ingredient_name):
    """Create a careful match key for one ingredient.

    This function is stricter than token overlap. It keeps fruit juices distinct
    and treats explicitly fresh citrus juice as fresh fruit.
    """
    normalized_without_details = normalize_ingredient_name(ingredient_name)

    if not normalized_without_details:
        return ""

    fruit_before_juice = extract_fruit_before_juice(normalized_without_details)

    if fruit_before_juice:
        if has_fresh_preparation_marker(ingredient_name):
            return f"fresh {fruit_before_juice}"
        return f"{fruit_before_juice} juice"

    if normalized_without_details in FRESH_FRUIT_WORDS:
        return f"fresh {normalized_without_details}"

    if normalized_without_details.startswith("fresh "):
        possible_fruit = normalized_without_details.replace("fresh ", "", 1).strip()
        if possible_fruit in FRESH_FRUIT_WORDS:
            return f"fresh {possible_fruit}"

    alias_key = canonicalize_alias_ingredient(normalized_without_details)

    if alias_key:
        return alias_key

    return normalized_without_details


def create_display_ingredient_name(ingredient_name):
    """Create a human-readable ingredient name for matching output and shopping lists."""
    normalized_without_details = normalize_ingredient_name(ingredient_name)
    fruit_before_juice = extract_fruit_before_juice(normalized_without_details)

    if fruit_before_juice and has_fresh_preparation_marker(ingredient_name):
        return f"fresh {fruit_before_juice} (for fresh juice)"

    return clean_text(ingredient_name)


def parse_ingredients_value(ingredients_value):
    """Parse one raw ingredients value into a list of dictionaries."""
    if ingredients_value is None:
        return []

    if isinstance(ingredients_value, list):
        ingredient_pairs = ingredients_value
    else:
        if pd.isna(ingredients_value) or str(ingredients_value).strip() == "":
            return []

        try:
            ingredient_pairs = ast.literal_eval(str(ingredients_value))
        except (ValueError, SyntaxError):
            ingredient_pairs = []

    parsed_ingredients = []

    for ingredient_pair in ingredient_pairs:
        if not isinstance(ingredient_pair, (list, tuple)) or len(ingredient_pair) < 2:
            continue

        amount_text = clean_text(ingredient_pair[0])
        ingredient_text = clean_text(ingredient_pair[1])
        normalized_name = normalize_ingredient_name(ingredient_text)
        match_key = create_ingredient_match_key(ingredient_text)
        display_name = create_display_ingredient_name(ingredient_text)

        if match_key:
            parsed_ingredients.append({
                "amount": amount_text,
                "ingredient": ingredient_text,
                "normalized_ingredient": normalized_name,
                "match_key": match_key,
                "display_ingredient": display_name,
            })

    return parsed_ingredients


def extract_unique_field(parsed_ingredients, field_name):
    """Extract one field from parsed ingredient dictionaries while removing duplicates."""
    values = [
        ingredient[field_name]
        for ingredient in parsed_ingredients
        if ingredient.get(field_name)
    ]
    return list(dict.fromkeys(values))


def build_search_text(row):
    """Create a readable text field that combines important recipe information."""
    ingredient_text = ", ".join(row["ingredient_display_names"])
    text_parts = [
        f"Title: {row['title']}",
        f"Glass: {row['glass']}",
        f"Garnish: {row['garnish']}",
        f"Ingredients: {ingredient_text}",
        f"Instructions: {row['recipe']}",
    ]
    return "\n".join(text_parts)


def clean_cocktail_dataframe(raw_dataframe):
    """Clean the raw cocktail recipe DataFrame."""
    cleaned_dataframe = raw_dataframe.copy()

    # Clean text columns without changing the original DataFrame.
    for column_name in ["title", "glass", "garnish", "recipe", "ingredients"]:
        if column_name in cleaned_dataframe.columns:
            cleaned_dataframe[column_name] = cleaned_dataframe[column_name].apply(clean_text)

    # Keep rows that have the minimum information needed for recommendation.
    cleaned_dataframe = cleaned_dataframe[
        (cleaned_dataframe["title"] != "")
        & (cleaned_dataframe["recipe"] != "")
        & (cleaned_dataframe["ingredients"] != "")
    ].copy()

    # Parse ingredients into structured lists.
    cleaned_dataframe["parsed_ingredients"] = cleaned_dataframe["ingredients"].apply(parse_ingredients_value)
    cleaned_dataframe["ingredient_names"] = cleaned_dataframe["parsed_ingredients"].apply(
        lambda ingredients: extract_unique_field(ingredients, "normalized_ingredient")
    )
    cleaned_dataframe["ingredient_match_keys"] = cleaned_dataframe["parsed_ingredients"].apply(
        lambda ingredients: extract_unique_field(ingredients, "match_key")
    )
    cleaned_dataframe["ingredient_display_names"] = cleaned_dataframe["parsed_ingredients"].apply(
        lambda ingredients: extract_unique_field(ingredients, "display_ingredient")
    )
    cleaned_dataframe["ingredient_count"] = cleaned_dataframe["ingredient_match_keys"].apply(len)

    # Remove rows where ingredients could not be parsed.
    cleaned_dataframe = cleaned_dataframe[cleaned_dataframe["ingredient_count"] > 0].copy()

    # Remove duplicate cocktail titles while keeping the first version.
    cleaned_dataframe = cleaned_dataframe.drop_duplicates(subset="title", keep="first")

    # Add a simple stable ID for later lookup.
    cleaned_dataframe = cleaned_dataframe.reset_index(drop=True)
    cleaned_dataframe["recipe_id"] = cleaned_dataframe.index

    # Build reusable recipe text for future document creation.
    cleaned_dataframe["search_text"] = cleaned_dataframe.apply(build_search_text, axis=1)

    return cleaned_dataframe


# Confirm the raw dataset exists from Section 2.
if "cocktails_raw_df" not in globals():
    raise NameError("Please run Section 2 first so cocktails_raw_df is available.")


# Create the cleaned dataset used by all later sections.
cocktails_clean_df = clean_cocktail_dataframe(cocktails_raw_df)

# Show what changed after cleaning.
print("Raw dataset shape:", cocktails_raw_df.shape)
print("Cleaned dataset shape:", cocktails_clean_df.shape)
print("Rows removed:", len(cocktails_raw_df) - len(cocktails_clean_df))

print("\nCleaned columns:")
print(list(cocktails_clean_df.columns))

print("\nSample cleaned ingredient data:")
display(cocktails_clean_df[[
    "recipe_id",
    "title",
    "ingredient_display_names",
    "ingredient_match_keys",
    "ingredient_count",
]].head(5))

## Short Comments

- We keep `cocktails_raw_df` unchanged so we can always return to the original data.
- We create `cocktails_clean_df` as the reliable working dataset.
- We parse ingredients into dictionaries because later functions need ingredient names separately from measurements.
- We create `ingredient_match_keys` so broad words like `juice` do not cause false matches.
- We treat explicitly fresh juice requirements as fresh fruit, such as `fresh lemon`, instead of bottled juice.

# Section 5: Ingredient Matching

This section builds the custom Ingredient Matching Engine.

Given a list of ingredients a user has at home, the engine calculates for every cocktail:

- matched ingredients
- missing ingredients
- match percentage

The matching now uses careful ingredient keys from Section 4. This prevents false matches such as `passion fruit juice` matching `lime juice`, and it keeps `fresh lemon` separate from bottled `lemon juice`.

## Why This Step Is Necessary

A home mixologist should not recommend drinks only because they sound similar to a query.

It should also answer a practical question:

```text
What can I make with what I already have?
```

This matching engine gives the project a rule-based recommendation layer before we add vector search and LLM prompting.

Careful ingredient identity is important. In cocktails, `apple juice`, `passion fruit juice`, `fresh lemon`, and `orange cura?ao liqueur` are different materials even if some of their words overlap.

In [ ]:
# ============================================================
# Section 5: Ingredient Matching
# ============================================================

import pandas as pd


def parse_available_ingredients(available_ingredients):
    """Convert user-provided ingredients into display names and strict match keys."""
    parsed_available_ingredients = []

    for available_ingredient in available_ingredients:
        ingredient_text = clean_text(available_ingredient)
        ingredient_key = create_ingredient_match_key(ingredient_text)

        if ingredient_key:
            parsed_available_ingredients.append({
                "user_text": ingredient_text,
                "match_key": ingredient_key,
            })

    return parsed_available_ingredients


def ingredient_key_is_available(recipe_match_key, parsed_available_ingredients):
    """Check whether a recipe ingredient key exists in the user's available keys."""
    available_keys = {
        ingredient["match_key"]
        for ingredient in parsed_available_ingredients
    }
    return recipe_match_key in available_keys


def get_recipe_ingredient_items(cocktail_row):
    """Return unique recipe ingredient items with both display names and match keys."""
    recipe_items = []
    seen_match_keys = set()

    for ingredient in cocktail_row["parsed_ingredients"]:
        match_key = ingredient.get("match_key", "")
        display_name = ingredient.get("display_ingredient", ingredient.get("ingredient", ""))

        if match_key and match_key not in seen_match_keys:
            recipe_items.append({
                "display_ingredient": display_name,
                "match_key": match_key,
            })
            seen_match_keys.add(match_key)

    return recipe_items


def match_single_cocktail(cocktail_row, available_ingredients):
    """Calculate ingredient matching details for one cocktail."""
    parsed_available_ingredients = parse_available_ingredients(available_ingredients)
    recipe_items = get_recipe_ingredient_items(cocktail_row)

    matched_ingredients = []
    missing_ingredients = []
    matched_ingredient_keys = []
    missing_ingredient_keys = []

    for recipe_item in recipe_items:
        recipe_key = recipe_item["match_key"]
        display_name = recipe_item["display_ingredient"]

        if ingredient_key_is_available(recipe_key, parsed_available_ingredients):
            matched_ingredients.append(display_name)
            matched_ingredient_keys.append(recipe_key)
        else:
            missing_ingredients.append(display_name)
            missing_ingredient_keys.append(recipe_key)

    total_ingredients = len(recipe_items)
    matched_count = len(matched_ingredients)
    missing_count = len(missing_ingredients)

    if total_ingredients == 0:
        match_percentage = 0.0
    else:
        match_percentage = round((matched_count / total_ingredients) * 100, 2)

    return {
        "recipe_id": cocktail_row["recipe_id"],
        "title": cocktail_row["title"],
        "glass": cocktail_row["glass"],
        "total_ingredients": total_ingredients,
        "matched_count": matched_count,
        "missing_count": missing_count,
        "match_percentage": match_percentage,
        "matched_ingredients": matched_ingredients,
        "missing_ingredients": missing_ingredients,
        "matched_ingredient_keys": matched_ingredient_keys,
        "missing_ingredient_keys": missing_ingredient_keys,
    }


def rank_cocktails_by_ingredients(clean_dataframe, available_ingredients, top_n=10):
    """Rank cocktails by how well they match the user's available ingredients."""
    match_rows = []

    for _, cocktail_row in clean_dataframe.iterrows():
        match_rows.append(match_single_cocktail(cocktail_row, available_ingredients))

    match_results_df = pd.DataFrame(match_rows)
    match_results_df = match_results_df.sort_values(
        by=["match_percentage", "missing_count", "matched_count", "title"],
        ascending=[False, True, False, True],
    ).reset_index(drop=True)

    if top_n is None:
        return match_results_df

    return match_results_df.head(top_n)


# Confirm the cleaned dataset exists from Section 4.
if "cocktails_clean_df" not in globals():
    raise NameError("Please run Section 4 first so cocktails_clean_df is available.")


# Example home bar ingredients for the teaching demo.
# Fresh citrus is written as fresh fruit because recipes such as
# "Lemon juice (freshly squeezed)" require fresh lemon, not bottled juice.
user_available_ingredients = [
    "gin",
    "rum",
    "vodka",
    "fresh lime",
]

# Rank the top cocktail matches for this example home bar.
matching_results_df = rank_cocktails_by_ingredients(
    cocktails_clean_df,
    user_available_ingredients,
    top_n=10,
)

print("Available ingredients:")
print(user_available_ingredients)

print("\nAvailable ingredient match keys:")
print([ingredient["match_key"] for ingredient in parse_available_ingredients(user_available_ingredients)])

print("\nTop matching cocktails:")
display(matching_results_df[[
    "title",
    "match_percentage",
    "matched_count",
    "missing_count",
    "matched_ingredients",
    "missing_ingredients",
]])

## Short Comments

- We compare strict ingredient match keys instead of loose word overlap.
- We calculate both matched and missing ingredients so recommendations are explainable.
- We rank by `match_percentage` first because the easiest cocktails should appear first.
- We distinguish fresh citrus requirements from bottled juices.
- This is a custom matching engine, not a LangChain or framework-based component.

# Section 6: Convert Recipes into Documents

This section converts each cleaned cocktail recipe into a text document.

In a manual RAG pipeline, a document is simply a searchable piece of text. Each cocktail becomes one document containing:

- title
- glass type
- garnish
- ingredients
- instructions

## Why This Step Is Necessary

Embedding models do not work directly with messy DataFrame rows.

They work best with readable text.

By converting each recipe into a document, we prepare the dataset for semantic search in later sections.

In [ ]:
# ============================================================
# Section 6: Convert Recipes into Documents
# ============================================================

import pandas as pd


def format_ingredient_lines(parsed_ingredients):
    """Format parsed ingredients into readable document lines."""
    ingredient_lines = []

    for ingredient in parsed_ingredients:
        amount = ingredient.get("amount", "")
        ingredient_name = ingredient.get("ingredient", "")

        if amount:
            ingredient_lines.append(f"- {amount} {ingredient_name}")
        else:
            ingredient_lines.append(f"- {ingredient_name}")

    return "\n".join(ingredient_lines)


def create_recipe_document(cocktail_row):
    """Convert one cleaned cocktail row into a document dictionary."""
    ingredient_lines = format_ingredient_lines(cocktail_row["parsed_ingredients"])

    document_text = f"""
Cocktail: {cocktail_row['title']}
Glass: {cocktail_row['glass']}
Garnish: {cocktail_row['garnish']}

Ingredients:
{ingredient_lines}

Instructions:
{cocktail_row['recipe']}
""".strip()

    return {
        "document_id": int(cocktail_row["recipe_id"]),
        "title": cocktail_row["title"],
        "text": document_text,
        "ingredient_names": cocktail_row["ingredient_display_names"],
        "ingredient_match_keys": cocktail_row["ingredient_match_keys"],
    }


def create_recipe_documents(clean_dataframe):
    """Create one searchable document for every cocktail recipe."""
    recipe_documents = []

    for _, cocktail_row in clean_dataframe.iterrows():
        recipe_documents.append(create_recipe_document(cocktail_row))

    return recipe_documents


# Confirm cleaned data exists from Section 4.
if "cocktails_clean_df" not in globals():
    raise NameError("Please run Section 4 first so cocktails_clean_df is available.")


# Convert cleaned cocktail rows into plain text documents.
recipe_documents = create_recipe_documents(cocktails_clean_df)
recipe_documents_df = pd.DataFrame(recipe_documents)

print(f"Created {len(recipe_documents)} recipe documents.")
print("\nSample document:")
print(recipe_documents[0]["text"][:1000])

## Short Comments

- We create one document per cocktail because each cocktail is a natural retrieval unit.
- We keep `document_id` aligned with `recipe_id` so search results can link back to the DataFrame.
- We include ingredient amounts in the document for better final recommendations.
- This is the document creation step of the manual RAG pipeline.

# Section 7: Generate Embeddings

This section turns recipe documents into embeddings.

An embedding is a list of numbers that represents the meaning of a piece of text. Similar texts should have similar embeddings.

Following the simpler RAG notebook style, we keep this section direct: load one embedding model, collect document texts, encode them, and store the result as a NumPy matrix.

## Why This Step Is Necessary

RAG systems need a way to find relevant documents.

Keyword search only matches exact words. Embedding search can match meaning.

For example, a query about "citrus gin drink" may retrieve recipes with fresh lemon, fresh lime, and gin even if the exact wording is different.

In [ ]:
# ============================================================
# Section 7: Generate Embeddings
# ============================================================

import numpy as np
from sentence_transformers import SentenceTransformer


EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"

# Confirm recipe documents exist from Section 6.
if "recipe_documents" not in globals():
    raise NameError("Please run Section 6 first so recipe_documents is available.")


# 1. Load a compact sentence-transformers embedding model.
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

# 2. Collect the text from every recipe document.
document_texts = [document["text"] for document in recipe_documents]

# 3. Convert the document text into normalized embeddings.
document_embeddings = embedding_model.encode(
    document_texts,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

# 4. FAISS expects float32 vectors.
document_embeddings = np.array(document_embeddings).astype("float32")

print("Embedding model:", EMBEDDING_MODEL_NAME)
print("Number of documents:", len(document_texts))
print("Embedding matrix shape:", document_embeddings.shape)
print("Embedding data type:", document_embeddings.dtype)

## Short Comments

- We embed the full recipe document, not just the cocktail title.
- We normalize embeddings so inner product search behaves like cosine similarity.
- We store embeddings in `float32` because FAISS expects efficient numeric arrays.
- This is the embeddings step of the manual RAG pipeline.

# Section 8: Create FAISS Index

This section creates a FAISS vector index.

FAISS stores document embeddings and lets us quickly search for the most similar documents to a user query.

## Why This Step Is Necessary

Embeddings are useful only if we can search them efficiently.

A vector index lets the notebook retrieve relevant cocktail documents quickly, even when there are thousands of recipes.

We use `IndexFlatIP` because our embeddings are normalized, so inner product acts like cosine similarity.

In [ ]:
# ============================================================
# Section 8: Create FAISS Index
# ============================================================

import faiss

# Confirm embeddings exist from Section 7.
if "document_embeddings" not in globals():
    raise NameError("Please run Section 7 first so document_embeddings is available.")


# The embedding dimension is the number of values in each vector.
embedding_dimension = document_embeddings.shape[1]

# Because embeddings are normalized, inner product works like cosine similarity.
faiss_index = faiss.IndexFlatIP(embedding_dimension)
faiss_index.add(document_embeddings)

print("FAISS index created.")
print("Number of vectors in index:", faiss_index.ntotal)
print("Embedding dimension:", embedding_dimension)

## Short Comments

- FAISS stores the embedding vectors for fast similarity search.
- `IndexFlatIP` is simple and beginner-friendly because it searches all vectors exactly.
- We use normalized embeddings, so higher scores mean more similar documents.
- This is the vector search index step of the manual RAG pipeline.

# Section 9: RAG Retrieval

This section performs the retrieval part of RAG manually.

The retrieval process is:

1. Convert the user query into an embedding.
2. Search the FAISS index.
3. Return the most similar cocktail documents.
4. Format those documents as context for the LLM.

## Why This Step Is Necessary

An LLM should not recommend from memory alone.

Retrieval gives the LLM grounded recipe information from our cocktail dataset.

This makes recommendations more relevant and easier to explain.

In [ ]:
# ============================================================
# Section 9: RAG Retrieval
# ============================================================

import numpy as np
import pandas as pd


def retrieve_relevant_documents(query_text, embedding_model, faiss_index, recipe_documents, top_k=5):
    """Embed a query, search FAISS, and return the most relevant recipe documents."""
    query_embedding = embedding_model.encode(
        [query_text],
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    query_embedding = np.array(query_embedding).astype("float32")

    similarity_scores, document_indices = faiss_index.search(query_embedding, top_k)
    retrieved_rows = []
    for rank, document_index in enumerate(document_indices[0], start=1):
        document = recipe_documents[int(document_index)]
        retrieved_rows.append({
            "rank": rank,
            "document_id": document["document_id"],
            "title": document["title"],
            "similarity_score": round(float(similarity_scores[0][rank - 1]), 4),
            "text": document["text"],
        })
    return pd.DataFrame(retrieved_rows)


def build_rag_context(retrieved_documents_df, max_characters_per_document=1200):
    """Combine only retrieved recipe documents into one prompt context block."""
    context_blocks = []
    for _, document_row in retrieved_documents_df.iterrows():
        document_text = document_row["text"][:max_characters_per_document]
        context_blocks.append(
            f"Document {document_row['rank']} - {document_row['title']}\n{document_text}"
        )
    return "\n\n---\n\n".join(context_blocks)


required_retrieval_objects = ["embedding_model", "faiss_index", "recipe_documents"]
missing_objects = [name for name in required_retrieval_objects if name not in globals()]
if missing_objects:
    raise NameError(f"Please run Sections 7-9 first. Missing: {missing_objects}")


sample_user_query = "Recommend a refreshing cocktail with citrus and herbs."
retrieved_context_df = retrieve_relevant_documents(
    sample_user_query, embedding_model, faiss_index, recipe_documents, top_k=5
)
rag_context = build_rag_context(retrieved_context_df)

print("User query:")
print(sample_user_query)
print("\nRetrieved documents:")
display(retrieved_context_df[["rank", "title", "similarity_score"]])
print("\nRAG context preview:")
print(rag_context[:1500])


## Short Comments

- We embed the user query with the same model used for recipe documents.
- FAISS returns document indices and similarity scores.
- We convert retrieved rows into text context for the LLM.
- This follows the simple RAG pattern: embed question, search index, build context, then prompt.

# Section 10: Qwen Instruct as a Visible Grounded Response Generator

This section adds `Qwen/Qwen3-4B-Instruct-2507` so its role is visible in each
chatbot reply. Qwen receives only selected recipe records or Python-calculated
inventory results. It writes a short English explanation above the recipe cards.

**Two visible answer layers**

1. **Qwen Instruct grounded explanation:** natural-language organisation and cautious interpretation of supplied evidence.
2. **Verified records:** titles, ingredients, quantities, glass, garnish, methods, and retrieval scores rendered directly by Python.


## The Grounding Rule

`LOAD_LLM` is `True` by default in this version. Run the notebook on a Colab GPU.

- Qwen may explain why a retrieved recipe is a *possible* match for a preference.
- Qwen may not invent titles, ingredients, quantities, methods, origins, or food-pairing claims.
- Python keeps the original records visible underneath Qwen's text.
- If Qwen cannot load, the notebook explicitly reports a deterministic RAG fallback.


In [ ]:
# ============================================================
# Section 10: Qwen Instruct in 4-bit mode
# ============================================================

# In Colab select Runtime > Change runtime type > T4 GPU (or a stronger GPU).
LOAD_LLM = True
PRIMARY_MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"
MAX_NEW_TOKENS = 220

import importlib.util
import platform
import torch


def show_runtime_report():
    print("Python platform:", platform.platform())
    print("PyTorch version:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        print("GPU memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))


def load_qwen_instruct():
    """Load the requested Qwen Instruct model in CUDA 4-bit NF4 mode."""
    if not torch.cuda.is_available():
        return None, None, None, "No CUDA GPU found: deterministic RAG fallback."

    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    try:
        tokenizer = AutoTokenizer.from_pretrained(PRIMARY_MODEL_NAME)
        model = AutoModelForCausalLM.from_pretrained(
            PRIMARY_MODEL_NAME,
            quantization_config=quantization_config,
            device_map="auto",
        )
        model.eval()
        return model, tokenizer, PRIMARY_MODEL_NAME, "Qwen loaded in 4-bit NF4 mode."
    except Exception as error:
        return None, None, None, f"Qwen load failed ({type(error).__name__}: {error}). Deterministic RAG fallback."


show_runtime_report()
if LOAD_LLM:
    if importlib.util.find_spec("bitsandbytes") is None:
        raise ImportError("Run the installation cell again so bitsandbytes is available for 4-bit Qwen loading.")
    llm_model, llm_tokenizer, loaded_model_name, loading_method = load_qwen_instruct()
else:
    llm_model, llm_tokenizer, loaded_model_name = None, None, None
    loading_method = "LLM deliberately disabled: deterministic RAG mode."

LLM_READY = llm_model is not None and llm_tokenizer is not None
print("Qwen ready:", LLM_READY)
print("Active generator:", loaded_model_name or "None")
print("Status:", loading_method)


## Colab Setup and Expected Behaviour

Use a Colab GPU with approximately **12 GB VRAM or more**. On a T4 GPU, Qwen is
loaded in 4-bit NF4 mode. The first run downloads the model and can take several
minutes.

This notebook uses only `Qwen/Qwen3-4B-Instruct-2507`; it does not silently
switch to another generator. If Qwen fails to load, Embedding + FAISS retrieval,
recipe lookup, and ingredient matching still work, but the interface labels the
answer as a deterministic fallback.


## What Qwen Does Here

- Qwen receives the user question plus only retrieved recipe records or deterministic inventory results.
- It writes the section headed **Qwen Instruct grounded explanation**.
- It must treat preference matching as an ingredient-based interpretation, not a verified flavour score or cultural claim.
- Python then shows unchanged source records below Qwen's text. Qwen does not select the FAISS ranking or calculate ingredient feasibility.


# Section 11: Ask the Grounded Cocktail Assistant

After running Sections 1-11 in order, use:

```python
ask_cocktail_assistant(question, available_ingredients=None, use_llm=True)
```

With a Colab GPU, `use_llm=True` makes Qwen write a visible grounded explanation.
The same response includes deterministic evidence records so factual recipe
content remains checkable.


## Grounding Boundaries

Embedding + FAISS retrieves recipe records before Qwen is called. Qwen receives
no hidden cocktail knowledge base and no unretrieved recipes.

Qwen may organise the answer and make a cautious ingredient-based interpretation.
It must not claim that a recipe is authentically regional, traditionally paired
with a food, or objectively refreshing unless that fact occurs in the evidence.

Food-pairing questions are also rejected in this version because the cocktail dataset does not contain sourced evidence linking dishes to cocktails.


In [ ]:
# ============================================================
# Section 11: One grounded assistant function
# ============================================================

from IPython.display import Markdown, display
import re
import time
import unicodedata


# Named recipe questions need a stronger threshold than broad preference
# searches. Preference queries retrieve a larger pool and expose all scores.
SEMANTIC_MIN_SCORE = 0.42
PREFERENCE_MIN_SCORE = 0.20


def normalize_title(title):
    """Make title matching case-insensitive and punctuation-insensitive."""
    title = unicodedata.normalize("NFKD", str(title)).encode("ascii", "ignore").decode("ascii")
    title = re.sub(r"[^a-z0-9]+", " ", title.lower())
    return re.sub(r"\s+", " ", title).strip()


def title_match_keys(title):
    """Create a narrow parent-name alias such as ``Mojito Cocktail`` -> ``Mojito``."""
    normalized = normalize_title(title)
    keys = [normalized]
    for optional_suffix in (" cocktail", " drink"):
        if normalized.endswith(optional_suffix):
            shortened = normalized[:-len(optional_suffix)].strip()
            if len(shortened) >= 4:
                keys.append(shortened)
    return list(dict.fromkeys(key for key in keys if len(key) >= 4))


title_to_document = {}
for document in recipe_documents:
    for title_key in title_match_keys(document["title"]):
        title_to_document.setdefault(title_key, document)

known_ingredient_tokens = {
    token
    for document in recipe_documents
    for match_key in document.get("ingredient_match_keys", [])
    for token in match_key.split()
    if len(token) >= 3 and token not in {"fresh"}
}


def has_recognized_recipe_signal(question):
    """Reject generic or fictional semantic recipe queries before FAISS can overmatch."""
    question_tokens = set(normalize_title(question).split())
    style_tokens = {"citrus", "refreshing", "summer", "light", "classic", "fizzy", "sour", "tropical", "herbs", "coffee", "ginger"}
    return bool(question_tokens & (known_ingredient_tokens | style_tokens))


def contains_fictional_ingredient_marker(question):
    """Reject clearly fictional ingredients before a generic vector match occurs."""
    normalized_question = normalize_title(question)
    fictional_markers = {
        "moon dust", "alien nectar", "starlight", "unicorn tears",
        "dragon blood", "magic potion",
    }
    return any(marker in normalized_question for marker in fictional_markers)


def find_exact_title_in_question(question):
    """Return a dataset recipe if its title appears inside a natural question."""
    normalized_question = f" {normalize_title(question)} "
    matching_titles = [
        title_key for title_key in title_to_document
        if f" {title_key} " in normalized_question
    ]
    if not matching_titles:
        return None
    return title_to_document[max(matching_titles, key=len)]


def recipe_row_from_document(document):
    """Recover the structured cleaned row behind a retrieved document."""
    recipe_id = int(document["document_id"])
    return cocktails_clean_df.loc[cocktails_clean_df["recipe_id"] == recipe_id].iloc[0]


def format_recipe_facts(document):
    """Format one retrieved dataset record without asking the LLM to rewrite it."""
    row = recipe_row_from_document(document)
    ingredient_lines = []
    for ingredient in row["parsed_ingredients"]:
        amount = ingredient.get("amount", "")
        name = ingredient.get("ingredient", "")
        ingredient_lines.append(f"- {amount} {name}".strip())
    parts = [f"## {row['title']}"]
    if row["glass"]:
        parts.append(f"**Glass:** {row['glass']}")
    if row["garnish"]:
        parts.append(f"**Garnish:** {row['garnish']}")
    parts.extend(["**Ingredients:**", "\n".join(ingredient_lines), "**Instructions:**", row["recipe"]])
    return "\n\n".join(parts)


def document_contains_term(document, term):
    """Check only normalized source ingredient evidence for an explicit query term."""
    evidence = " ".join(document.get("ingredient_match_keys", [])).lower()
    return bool(re.search(rf"(?<!\w){re.escape(term)}(?!\w)", evidence))


def explicit_preference_constraints(question):
    """Extract only verifiable ingredient constraints, not flavour tags."""
    text = normalize_title(question)
    required = {term for term in {"gin", "ginger", "lime", "coffee"} if re.search(rf"(?<!\w){term}(?!\w)", text)}
    excluded = {"cream", "milk"} if re.search(r"\b(?:not creamy|without cream|no cream)\b", text) else set()
    return required, excluded


def retrieve_semantic_preferences(question, top_k=15, top_n=3):
    """Use Embedding + FAISS for a descriptive cocktail preference.

    FAISS supplies the candidate order. Explicit user constraints only remove
    contradictions; no manually authored refreshing, tropical, or regional
    score is used.
    """
    if contains_fictional_ingredient_marker(question):
        return [], pd.DataFrame()
    candidates = retrieve_relevant_documents(
        question, embedding_model, faiss_index, recipe_documents, top_k=top_k
    )
    required, excluded = explicit_preference_constraints(question)
    selected_rows = []
    selected_documents = []
    for _, candidate in candidates.iterrows():
        if float(candidate["similarity_score"]) < PREFERENCE_MIN_SCORE:
            continue
        document = next(item for item in recipe_documents if item["document_id"] == candidate["document_id"])
        if not all(document_contains_term(document, term) for term in required):
            continue
        if any(document_contains_term(document, term) for term in excluded):
            continue
        selected_documents.append(document)
        selected_rows.append({
            "rank": len(selected_documents),
            "title": document["title"],
            "retrieval_type": "semantic_faiss",
            "similarity_score": float(candidate["similarity_score"]),
            "document_id": document["document_id"],
            "text": document["text"],
        })
        if len(selected_documents) == top_n:
            break
    return selected_documents, pd.DataFrame(selected_rows)


def is_unsupported_food_pairing_request(text):
    """Detect food-pairing requests that this recipe-only dataset cannot verify."""
    pairing_terms = {"pair", "pairs", "pairing", "match", "matches", "matching", "accompany", "accompanies"}
    food_terms = {
        "food", "dish", "meal", "dinner", "lunch", "chicken", "rice", "steak", "sushi",
        "pizza", "noodle", "noodles", "burger", "curry", "fish", "seafood", "pasta", "salad", "dessert",
    }
    tokens = set(text.split())
    return bool(tokens & pairing_terms) and bool(tokens & food_terms)


def infer_intent(question, available_ingredients=None):
    """Route with transparent rules; descriptive recommendations must use RAG."""
    text = normalize_title(question)
    if is_unsupported_food_pairing_request(text):
        return "food_pairing_unsupported"
    if available_ingredients:
        return "ingredient_match"
    if any(phrase in text for phrase in ["i have", "my bar", "what can i make", "available ingredients", "makeable"]):
        return "ingredient_match"
    if any(phrase in text for phrase in ["how do i make", "recipe for", "ingredients for", "ingredients are needed", "prepare"]):
        return "named_recipe"
    if any(phrase in text for phrase in ["recommend", "suggest", "i want", "find", "something", "refreshing", "tropical", "fizzy", "coffee", "ginger", "lime", "summer", "light gin"]):
        return "semantic_preference"
    if find_exact_title_in_question(question) is not None:
        return "named_recipe"
    if any(word in text for word in ["cocktail", "drink", "recipe", "mix", "martini", "sour"]):
        return "semantic_recipe"
    return "out_of_scope"


def retrieve_one_grounded_recipe(question):
    """Use exact title first; otherwise use FAISS and reject weak matches."""
    if contains_fictional_ingredient_marker(question):
        return None, 0.0, "weak_retrieval"
    exact_document = find_exact_title_in_question(question)
    if exact_document is not None:
        return exact_document, 1.0, "exact_title"
    if not has_recognized_recipe_signal(question):
        return None, 0.0, "weak_retrieval"
    results = retrieve_relevant_documents(question, embedding_model, faiss_index, recipe_documents, top_k=1)
    top_score = float(results.iloc[0]["similarity_score"])
    if top_score < SEMANTIC_MIN_SCORE:
        return None, top_score, "weak_retrieval"
    document_id = int(results.iloc[0]["document_id"])
    document = next(item for item in recipe_documents if int(item["document_id"]) == document_id)
    return document, top_score, "semantic"


def build_qwen_evidence_context(documents):
    """Serialize only retrieved source records that Qwen may use."""
    blocks = []
    for rank, document in enumerate(documents, start=1):
        blocks.append(
            f"[Retrieved record {rank}]\nTitle: {document['title']}\n{document['text']}"
        )
    return "\n\n".join(blocks)


def build_inventory_evidence_context(matches, available_ingredients):
    """Serialize Python-calculated inventory results without asking Qwen to recalculate."""
    blocks = [f"User-provided ingredients: {', '.join(available_ingredients)}"]
    for rank, (_, match) in enumerate(matches.iterrows(), start=1):
        missing = ", ".join(match["missing_ingredients"]) or "Nothing"
        blocks.append(
            f"[Calculated result {rank}]\nTitle: {match['title']}\n"
            f"Match percentage: {match['match_percentage']}%\nMissing ingredients: {missing}"
        )
    return "\n\n".join(blocks)


def build_qwen_generation_prompt(question, intent, evidence_context):
    """Create a transparent, evidence-bounded prompt for Qwen Instruct."""
    task_by_intent = {
        "ingredient_match": "Explain the verified inventory results. Do not call a recipe fully makeable when it has missing ingredients.",
        "semantic_preference": "Explain why the retrieved cocktails are possible matches. Use cautious wording such as 'may suit' or 'possible match'.",
        "named_recipe": "Introduce the retrieved recipe and direct the user to the verified record below.",
        "semantic_recipe": "Introduce the retrieved recipe and direct the user to the verified record below.",
    }
    return (
        f"User question: {question}\n\n"
        f"Task: {task_by_intent.get(intent, 'Summarize the supplied cocktail evidence.')}\n\n"
        "Write 2 to 4 concise English sentences using only the evidence below. Every named cocktail must be a supplied title. "
        "Do not invent or alter ingredients, quantities, methods, glassware, garnish, origins, rankings, safety warnings, or food-pairing claims. "
        "For preference questions, an ingredient-based interpretation is allowed only when it names source ingredients or methods in the evidence. "
        "Do not call a result objectively refreshing, authentic, traditional, or verified unless the evidence says so. "
        "If evidence is insufficient, output exactly: INSUFFICIENT_EVIDENCE\n\n"
        f"Evidence:\n{evidence_context}"
    )


def generate_qwen_grounded_explanation(prompt):
    """Generate a visible Qwen explanation, or return an empty fallback signal."""
    if not LLM_READY:
        return ""
    messages = [
        {"role": "system", "content": "You are CocktailCompass, an evidence-grounded cocktail assistant. All output must be English. Follow the evidence boundary exactly."},
        {"role": "user", "content": prompt},
    ]
    try:
        try:
            model_inputs = llm_tokenizer.apply_chat_template(
                messages, tokenize=True, add_generation_prompt=True, enable_thinking=False,
                return_tensors="pt", return_dict=True,
            )
        except TypeError:
            model_inputs = llm_tokenizer.apply_chat_template(
                messages, tokenize=True, add_generation_prompt=True,
                return_tensors="pt", return_dict=True,
            )
        model_inputs = {name: value.to(llm_model.device) for name, value in model_inputs.items()}
        generated_ids = llm_model.generate(
            **model_inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
            temperature=None, top_p=None, repetition_penalty=1.05,
            pad_token_id=llm_tokenizer.eos_token_id,
        )
        new_tokens = generated_ids[0][model_inputs["input_ids"].shape[-1]:]
        explanation = llm_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    except Exception as error:
        print(f"Qwen generation skipped: {type(error).__name__}: {error}")
        return ""
    explanation = re.sub(r"<think>.*?</think>", "", explanation, flags=re.DOTALL).strip()
    if explanation == "INSUFFICIENT_EVIDENCE":
        return explanation
    return explanation if 2 <= len(explanation) <= 1400 else ""


def render_qwen_explanation(question, intent, documents=None, matches=None, available_ingredients=None, use_llm=True):
    """Show Qwen's distinct language layer before deterministic evidence cards."""
    if not use_llm:
        return ""
    if not LLM_READY:
        return "## Qwen Instruct grounded explanation\n\n_Qwen is unavailable. Deterministic RAG evidence is shown below._"
    context = (
        build_inventory_evidence_context(matches, available_ingredients)
        if intent == "ingredient_match"
        else build_qwen_evidence_context(documents)
    )
    explanation = generate_qwen_grounded_explanation(build_qwen_generation_prompt(question, intent, context))
    if explanation == "INSUFFICIENT_EVIDENCE":
        return "## Qwen Instruct grounded explanation\n\nQwen reported insufficient evidence for an added interpretation. Verified records are shown below."
    if not explanation:
        return "## Qwen Instruct grounded explanation\n\n_Qwen did not return a usable grounded explanation. Deterministic evidence is shown below._"
    return (
        "## Qwen Instruct grounded explanation\n\n"
        f"{explanation}\n\n"
        "> **Evidence boundary:** This text was generated only from the retrieved records. The verified records below remain the source of truth."
    )


def ask_cocktail_assistant(question, available_ingredients=None, use_llm=True):
    """Return a visible Qwen explanation plus deterministic recipe or inventory evidence."""
    question = str(question).strip()
    if not question:
        return {"status": "rejected", "reason": "empty_question", "answer": "Please enter a cocktail question."}
    intent = infer_intent(question, available_ingredients)
    if intent == "food_pairing_unsupported":
        return {
            "status": "rejected",
            "intent": intent,
            "reason": "missing_food_pairing_evidence",
            "answer": (
                "I cannot verify a food-pairing recommendation because this dataset contains cocktail recipes "
                "but no sourced evidence about which cocktails pair with that dish."
            ),
        }
    if intent == "out_of_scope":
        return {"status": "rejected", "intent": intent, "reason": "out_of_scope", "answer": "I can help with cocktail recipes, available-ingredient matching, or semantic cocktail recommendations."}
    if intent == "ingredient_match":
        if not available_ingredients:
            return {"status": "rejected", "intent": intent, "reason": "missing_ingredients", "answer": "Please provide available_ingredients, for example ['gin', 'fresh lime', 'soda water']."}
        matches = rank_cocktails_by_ingredients(cocktails_clean_df, available_ingredients, top_n=3)
        qwen_layer = render_qwen_explanation(question, intent, matches=matches, available_ingredients=available_ingredients, use_llm=use_llm)
        lines = ["## Verified inventory-match results"]
        for rank, (_, match) in enumerate(matches.iterrows(), start=1):
            missing = ", ".join(match["missing_ingredients"]) or "Nothing"
            lines.append(f"{rank}. **{match['title']}** - {match['match_percentage']}% match; missing: {missing}.")
        return {"status": "ok", "intent": intent, "answer": "\n\n".join(([qwen_layer] if qwen_layer else []) + lines), "evidence": matches}
    if intent == "semantic_preference":
        documents, evidence = retrieve_semantic_preferences(question, top_k=15, top_n=3)
        if not documents:
            return {"status": "rejected", "intent": intent, "reason": "insufficient_semantic_evidence", "answer": "I could not find sufficiently relevant cocktail evidence for that preference, so I will not invent a recommendation."}
        qwen_layer = render_qwen_explanation(question, intent, documents=documents, use_llm=use_llm)
        lines = ["## Verified retrieved recipe records", "The following records were retrieved with Embedding + FAISS. Cosine similarity ranks relevance to your wording; it is not a flavour score."]
        for rank, document in enumerate(documents, start=1):
            similarity = float(evidence.iloc[rank - 1]["similarity_score"])
            facts = format_recipe_facts(document).replace("## ", "### ", 1)
            lines.append(f"{facts}\n\n**FAISS cosine similarity:** {similarity:.4f}")
        return {"status": "ok", "intent": intent, "answer": "\n\n".join(([qwen_layer] if qwen_layer else []) + lines), "evidence": evidence}
    document, score, retrieval_type = retrieve_one_grounded_recipe(question)
    if document is None:
        return {"status": "rejected", "intent": intent, "reason": "weak_retrieval", "similarity_score": round(score, 4), "answer": "I could not find a sufficiently relevant cocktail record, so I will not invent a recipe."}
    qwen_layer = render_qwen_explanation(question, intent, documents=[document], use_llm=use_llm)
    evidence = pd.DataFrame([{"title": document["title"], "retrieval_type": retrieval_type, "similarity_score": round(score, 4), "document_id": document["document_id"]}])
    answer = "\n\n".join(([qwen_layer] if qwen_layer else []) + ["## Verified recipe record", format_recipe_facts(document)])
    return {"status": "ok", "intent": intent, "answer": answer, "evidence": evidence}


def display_assistant_answer(result):
    """Display an answer and its retrieved evidence in a consistent format."""
    display(Markdown(result["answer"]))
    if result.get("status") == "ok":
        print("\nRetrieved evidence:")
        display(result["evidence"])
    elif result.get("reason"):
        print("Reason:", result["reason"])


DEMO_QUESTION = "Recommend a cocktail with ginger and lime."
DEMO_INGREDIENTS = ["gin", "fresh lime", "soda water", "mint"]
demo_result = ask_cocktail_assistant(DEMO_QUESTION, use_llm=LLM_READY)
display_assistant_answer(demo_result)

print("\nIngredient-match demo:")
ingredient_demo_result = ask_cocktail_assistant("What can I make?", available_ingredients=DEMO_INGREDIENTS, use_llm=False)
display_assistant_answer(ingredient_demo_result)


## How to Use It

Examples:

```python
ask_cocktail_assistant("How do I make a Mojito?")
ask_cocktail_assistant("What can I make?", ["gin", "fresh lime", "soda water"])
ask_cocktail_assistant("Recommend a refreshing cocktail with citrus and herbs.")
ask_cocktail_assistant("I want something tropical and fizzy.")
ask_cocktail_assistant("Suggest a coffee-flavoured cocktail that is not creamy.")
ask_cocktail_assistant("Find a light gin cocktail for summer.")
ask_cocktail_assistant("Recommend a cocktail with ginger and lime.")
```


## Short Comments

- Exact recipe titles work with different capitalisation and inside natural questions.
- Available-ingredient matching uses deterministic source ingredient keys.
- Descriptive preferences use FAISS semantic retrieval, not a hand-authored refreshing score.
- The evidence table shows the retrieved titles, retrieval type, and cosine similarities.
- `display_assistant_answer(...)` always shows retrieved evidence after a successful answer.


# Section 12: Evaluation, Limitations, and Final Checks

This compact evaluation uses pandas DataFrames and prints actual results from the current runtime. It does not invent scores.


## What Is Evaluated?

- named-recipe retrieval
- available-ingredient matching using actual recipe ingredients
- five semantic preference queries that must use Embedding + FAISS
- visible cosine similarities and explicit contradiction checks
- similarity-threshold decisions
- FAISS retrieval time
- optional no-RAG versus grounded-RAG text comparison

The manual rubric stays marked as pending until a person reviews generated responses.


In [ ]:
# ============================================================
# Section 12: Small, reproducible evaluation
# ============================================================

import time


# 1. Named recipe retrieval: actual titles from this loaded dataset.
preferred_titles = ["Mojito", "Margarita", "Negroni", "Daiquiri", "Martini"]
available_titles = {normalize_title(document["title"]): document["title"] for document in recipe_documents}
evaluation_titles = [available_titles[normalize_title(title)] for title in preferred_titles if normalize_title(title) in available_titles]
if len(evaluation_titles) < 5:
    evaluation_titles.extend([document["title"] for document in recipe_documents if document["title"] not in evaluation_titles][:5 - len(evaluation_titles)])
named_rows = []
for title in evaluation_titles:
    result = ask_cocktail_assistant(f"How do I make {title}?", use_llm=False)
    retrieved_title = result["evidence"].iloc[0]["title"] if result["status"] == "ok" else None
    named_rows.append({"question": f"How do I make {title}?", "expected_title": title, "retrieved_title": retrieved_title, "passed": retrieved_title == title})
named_retrieval_evaluation_df = pd.DataFrame(named_rows)


# 2. Ingredient matching: each test uses ingredients from an actual record.
ingredient_rows = []
for document in recipe_documents[:3]:
    row = recipe_row_from_document(document)
    supplied = row["ingredient_match_keys"]
    matches = rank_cocktails_by_ingredients(cocktails_clean_df, supplied, top_n=None)
    own_match = matches[matches["recipe_id"] == row["recipe_id"]].iloc[0]
    ingredient_rows.append({"recipe": row["title"], "supplied_key_count": len(supplied), "missing_count": int(own_match["missing_count"]), "match_percentage": float(own_match["match_percentage"]), "passed": int(own_match["missing_count"]) == 0})
ingredient_matching_evaluation_df = pd.DataFrame(ingredient_rows)


# 3. Required semantic-RAG preference cases. Each must route through FAISS.
preference_queries = [
    "Recommend a refreshing cocktail with citrus and herbs.",
    "I want something tropical and fizzy.",
    "Suggest a coffee-flavoured cocktail that is not creamy.",
    "Find a light gin cocktail for summer.",
    "Recommend a cocktail with ginger and lime.",
]
preference_rows = []
for question in preference_queries:
    result = ask_cocktail_assistant(question, use_llm=False)
    evidence = result.get("evidence", pd.DataFrame())
    preference_rows.append({
        "question": question,
        "intent": result.get("intent"),
        "status": result["status"],
        "used_embedding_faiss": bool(not evidence.empty and (evidence["retrieval_type"] == "semantic_faiss").all()),
        "retrieved_titles": evidence["title"].tolist() if not evidence.empty else [],
        "cosine_similarities": evidence["similarity_score"].tolist() if not evidence.empty else [],
        "reason": result.get("reason"),
    })
semantic_preference_evaluation_df = pd.DataFrame(preference_rows)


# 4. Threshold experiment and timing measurement from this runtime.
threshold_queries = [
    "a refreshing citrus gin cocktail",
    "a cocktail with moon dust and alien nectar",
    "I want a drink with fresh mint and soda water",
]
threshold_rows = []
for question in threshold_queries:
    documents, evidence = retrieve_semantic_preferences(question, top_k=15, top_n=3)
    top_score = float(evidence.iloc[0]["similarity_score"]) if not evidence.empty else 0.0
    threshold_rows.append({"question": question, "top_similarity": round(top_score, 4), "decision_at_0_20": top_score >= 0.20, "retrieved_count": len(documents)})
threshold_experiment_df = pd.DataFrame(threshold_rows)

timing_start = time.perf_counter()
for _ in range(5):
    retrieve_relevant_documents("Recommend a refreshing cocktail with citrus and herbs.", embedding_model, faiss_index, recipe_documents, top_k=3)
retrieval_average_ms = (time.perf_counter() - timing_start) / 5 * 1000
efficiency_evaluation_df = pd.DataFrame([{"operation": "Embedding + FAISS top-3 retrieval", "runs": 5, "average_milliseconds": round(retrieval_average_ms, 2)}])


# 5. Optional no-RAG versus RAG comparison.
comparison_question = preference_queries[0]
rag_answer = ask_cocktail_assistant(comparison_question, use_llm=LLM_READY)["answer"]
if LLM_READY:
    no_rag_prompt = (
        "Evaluation baseline only: answer this cocktail preference without any retrieved recipe records. "
        "No evidence is supplied, so this output is intentionally ungrounded and must not be treated as a verified recommendation. "
        f"Question: {comparison_question}"
    )
    no_rag_answer = generate_qwen_grounded_explanation(no_rag_prompt)
    if not no_rag_answer:
        no_rag_answer = "No usable Qwen baseline output was generated."
else:
    no_rag_answer = "Not run: load Qwen on a Colab GPU to record this comparison."
rag_comparison_df = pd.DataFrame([{"question": comparison_question, "no_rag_output": no_rag_answer, "grounded_rag_output": rag_answer}])


# 6. A manual rubric intentionally has no invented scores.
manual_rubric_df = pd.DataFrame([
    {"case": "named recipe", "grounded": "pending manual review", "helpful": "pending manual review", "format_ok": "pending manual review"},
    {"case": "ingredient matching", "grounded": "pending manual review", "helpful": "pending manual review", "format_ok": "pending manual review"},
    {"case": "semantic preference RAG", "grounded": "pending manual review", "helpful": "pending manual review", "format_ok": "pending manual review"},
])


assert normalize_title("Mojíto!") == "mojito"
assert find_exact_title_in_question(f"Can you show me a {evaluation_titles[0]} recipe?") is not None
assert ask_cocktail_assistant("How can I repair a bicycle?")["status"] == "rejected"
assert ask_cocktail_assistant("Recommend a cocktail made with moon dust and alien nectar")["status"] == "rejected"
assert ingredient_matching_evaluation_df["passed"].all()
assert (semantic_preference_evaluation_df["intent"] == "semantic_preference").all()

print("Named-recipe retrieval evaluation")
display(named_retrieval_evaluation_df)
print("\nIngredient-matching evaluation")
display(ingredient_matching_evaluation_df)
print("\nSemantic preference RAG evaluation")
display(semantic_preference_evaluation_df)
print("\nSimilarity-threshold experiment")
display(threshold_experiment_df)
print("\nEfficiency measurement")
display(efficiency_evaluation_df)
print("\nManual generated-answer rubric")
display(manual_rubric_df)
print("\nNo-RAG versus grounded RAG")
display(rag_comparison_df)
print("\nAll Section 12 checks passed.")


## Interpreting the Results

The tables record this run's results, so rerun the cell after changing the
data, embedding model, or threshold. Cosine similarity reflects semantic
closeness to the query wording; it is not a verified flavour rating, regional
origin, or guarantee that the recipe satisfies an unspoken preference.

Known limitations: ingredient aliases are intentionally conservative; the
dataset can contain unusual or incomplete recipes; semantic similarity can
return weak matches for subjective words such as `refreshing` or `tropical`;
and Qwen needs a GPU for practical local use.


## Final Reproducibility Checklist

Run the notebook from top to bottom in one Colab session. No API key and no
separate Python files are required.

For the final submission, retain the visible evidence tables and example
outputs after running the notebook. If a GPU is unavailable, Embedding + FAISS
retrieval, available-ingredient matching, tests, and deterministic source
rendering still run; only the optional Qwen introduction and no-RAG comparison
remain unexecuted.


# Section 13: Interactive Chatbot Demo

Each successful reply visibly separates two RAG layers:

1. **Qwen Instruct grounded explanation** - natural English generated from retrieved evidence.
2. **Verified records** - deterministic recipe facts plus a compact retrieval-evidence table.

For available-ingredient matching, write what you have naturally in the same
message, for example: `I have gin, fresh lime, soda water and mint. What can I make?`


In [ ]:
# ============================================================
# Section 13: Interactive chatbot interface (Gradio)
# ============================================================

try:
    import gradio as gr
except ImportError as error:
    raise ImportError("Gradio is missing. Rerun Section 1, then rerun this cell.") from error


def extract_home_bar_ingredients(message):
    """Read ingredients from a natural home-bar sentence when possible."""
    message = str(message or "").replace("，", ",")
    match = re.search(r"\b(?:i have|ingredients?\s*[:=-])\s*(.+)", message, flags=re.IGNORECASE)
    if not match:
        return None
    ingredient_text = re.split(r"\b(?:what|which|can|could|please|recommend|suggest)\b", match.group(1), maxsplit=1, flags=re.IGNORECASE)[0]
    ingredients = [item.strip(" .?!,;:") for item in re.split(r"\s*,\s*|\s+and\s+|\s*;\s*", ingredient_text, flags=re.IGNORECASE) if item.strip(" .?!,;:")]
    return list(dict.fromkeys(ingredients)) or None


def format_chat_evidence(result):
    """Keep retrieval evidence visible in the chat without a wide table."""
    evidence = result.get("evidence")
    if isinstance(evidence, pd.DataFrame) and not evidence.empty:
        columns = [column for column in ["rank", "title", "retrieval_type", "similarity_score", "match_percentage", "missing_count"] if column in evidence.columns]
        preview = evidence[columns].head(3).to_string(index=False)
        return "\n\n---\n**Retrieved evidence**\n```text\n" + preview + "\n```"
    if result.get("reason"):
        return "\n\n---\n**Safety result:** " + str(result["reason"])
    return ""


def chat_response(message, history):
    """Function called by Gradio's ChatInterface for every user message."""
    available_ingredients = extract_home_bar_ingredients(message)
    result = ask_cocktail_assistant(message, available_ingredients=available_ingredients, use_llm=True)
    return result["answer"] + format_chat_evidence(result)


previous_demo = globals().get("cocktail_chatbot_demo")
if previous_demo is not None:
    previous_demo.close()

cocktail_chatbot_demo = gr.ChatInterface(
    fn=chat_response,
    title="AI Home Mixologist - Qwen Grounded Generation RAG",
    description="Qwen Instruct writes a clearly labelled explanation from Embedding + FAISS retrieved evidence. Verified records remain visible below it.",
    examples=[
        ["How do I make a Mojito?"],
        ["I have gin, fresh lime, soda water and mint. What can I make?"],
        ["Recommend a refreshing cocktail with citrus and herbs."],
        ["I want something tropical and fizzy."],
        ["Suggest a coffee-flavoured cocktail that is not creamy."],
        ["Find a light gin cocktail for summer."],
        ["Recommend a cocktail with ginger and lime."],
    ],
)

print("Launching the interactive assistant. Use the embedded chat interface below.")
cocktail_chatbot_demo.launch()


## Chatbot Demonstration Notes

Use a semantic preference such as:

```text
Recommend a refreshing cocktail with citrus and herbs.
```

For the presentation, point out the sequence: the question is embedded, FAISS
retrieves recipe records, Qwen receives only those records and writes the
explanation, then Python shows the original records and similarity scores.
